In [1]:
from peewee import PostgresqlDatabase
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [2]:
db = PostgresqlDatabase(
    'mydatabase', 
    user='myuser',  
    password='mysecretpassword',  
    host='localhost',  
    port=5432  
)

In [3]:
from db_manager.db_conf import db

class DBManager:
    
    @staticmethod
    def connect_db():
        db.connect()
        return db.connection() 

    @staticmethod
    def close_db():
        db.close()

    @staticmethod
    def create_tables(table):
        db.create_tables([table])

    @staticmethod
    def drop_tables(table):
        db.drop_tables([table])

    @staticmethod
    def create_schemas(schema_name):
        db.execute_sql(f'CREATE SCHEMA IF NOT EXISTS {schema_name};')

    @staticmethod
    def drop_schemas(schema_name):
        db.execute_sql(f'DROP SCHEMA IF EXISTS {schema_name} CASCADE;')

    @staticmethod
    def table_exists(table_name):
        return db.table_exists(table_name)
    
    @staticmethod
    def transaction():
        return db.atomic()

In [6]:
conn = DBManager.connect_db()

In [ ]:
query = """
select
	ps.id_player_id,
    pa.player,
    pa.player_effiencey_rating,
    pa.true_shooting_percentage,
    pa.total_rebound_percentage,
    pa.assist_percentage,
    pa.steal_percentage,
    pa.block_percentage,
    pa.turnover_percentage,
    pa.usage_percentage,
    pa.win_shares,
    pa.box_plus_minus,
    pa.value_over_replacement_player,
    ps.field_goals_percentage,
    ps.three_point_field_goals_percentage,
    ps.two_point_field_goals_percentage,
    ps.effective_field_goals_percentage,
    ps.free_throws_percentage,
    ps.total_rebounds
FROM 
    players.player_advanced pa
JOIN 
    players.player_stats ps 
ON 
    pa.id_player_id = ps.id_player_id
    AND pa."year" = ps."year"
    AND pa.minutes_played = ps.minutes_played
WHERE 
    pa.minutes_played > 300
	AND 
	pa.id_player_id IN (
        SELECT id_player_id
        FROM players.player_advanced
        GROUP BY id_player_id
        HAVING COUNT(DISTINCT "year") > 3
    );
"""

In [8]:
df = pd.read_sql(query, conn)
df = df.apply(pd.to_numeric, errors="coerce")

C:\Users\adhc_\AppData\Local\Temp\ipykernel_29452\3563067569.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [9]:
correlation_matrix = df.iloc[:, 1:].corr()
threshold = 0.7
high_corr = (correlation_matrix.abs() > threshold) & (correlation_matrix != 1)
correlated_pairs = correlation_matrix[high_corr].stack().reset_index()
correlated_pairs.columns = ["Variable 1", "Variable 2", "Correlación"]

In [10]:
correlated_pairs

,Variable 1,Variable 2,Correlación
0,player_effiencey_rating,win_shares,0.766302
1,player_effiencey_rating,box_plus_minus,0.851794
2,player_effiencey_rating,value_over_replacement_player,0.788084
3,true_shooting_percentage,field_goals_percentage,0.745096
4,true_shooting_percentage,two_point_field_goals_percentage,0.789390
5,true_shooting_percentage,effective_field_goals_percentage,0.929410
6,win_shares,player_effiencey_rating,0.766302
7,win_shares,box_plus_minus,0.802440
8,win_shares,value_over_replacement_player,0.915095
9,win_shares,total_rebounds,0.703008


In [ ]:
df_scale = df[[
    "true_shooting_percentage",
    "effective_field_goals_percentage",
    "value_over_replacement_player",
    "win_shares",
    "box_plus_minus",
    "player_effiencey_rating",
    "field_goals_percentage",
    "two_point_field_goals_percentage"
]].copy()

scaler = MinMaxScaler()
df_scale.iloc[:, 1:] = scaler.fit_transform(df_scale.iloc[:, 1:])

df_scale["overall"] = (
    df_scale["value_over_replacement_player"] * 0.20 +
    df_scale["win_shares"] * 0.20 +
    df_scale["player_effiencey_rating"] * 0.10 +
    df_scale["box_plus_minus"] * 0.15 +
    df_scale["effective_field_goals_percentage"] * 0.10 +
    df_scale["true_shooting_percentage"] * 0.05 +
    df_scale["two_point_field_goals_percentage"] * 0.05 +
    df_scale["field_goals_percentage"] * 0.05
)
df_scale["overall"] = MinMaxScaler().fit_transform(df_scale[["overall"]])
df_scale["overall"] = 50 + (df_scale["overall"] * (99 - 50))


In [24]:
df

,id_player_id,player_effiencey_rating,true_shooting_percentage,total_rebound_percentage,assist_percentage,steal_percentage,block_percentage,turnover_percentage,usage_percentage,win_shares,box_plus_minus,value_over_replacement_player,field_goals_percentage,three_point_field_goals_percentage,two_point_field_goals_percentage,effective_field_goals_percentage,free_throws_percentage,total_rebounds
0,11,19.4,0.621,21.2,5.4,0.5,5.9,18.7,13.8,9.9,2.4,3.3,0.562,NaN,0.562,0.562,0.708,1157
1,7,14.7,0.503,11.3,4.6,1.5,1.3,11.4,19.4,4.3,-1.6,0.3,0.461,0.100,0.465,0.462,0.671,571
2,8,14.3,0.496,7.8,14.5,1.1,0.2,12.2,24.2,2.8,-0.8,0.8,0.411,0.386,0.418,0.458,0.877,394
3,12,15.7,0.488,6.8,19.4,1.0,0.2,12.4,28.8,1.6,-1.4,0.3,0.419,0.311,0.441,0.445,0.785,258
4,3,12.9,0.494,4.9,24.6,1.6,0.4,13.6,17.4,2.3,-2.2,-0.1,0.455,0.205,0.472,0.461,0.817,172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14857,13041,16.7,0.618,15.2,2.5,1.2,3.9,15.7,18.1,5.3,-0.7,0.6,0.604,NaN,0.604,0.604,0.618,481
14858,2801,14.6,0.546,8.2,6.6,2.0,0.8,10.3,20.7,4.4,0.4,1.1,0.425,0.353,0.481,0.503,0.839,253
14859,20881,8.8,0.477,9.3,5.8,1.5,2.2,8.3,10.1,2.7,-0.9,0.4,0.411,0.315,0.449,0.455,0.769,256
14860,8791,10.1,0.520,6.8,8.3,2.1,1.1,14.3,16.6,1.6,-1.4,0.1,0.407,0.335,0.474,0.488,0.768,132


In [ ]:
DBManager.close_db()